# Bodhan Tool Server & Task Agent — Guided Walkthrough

This notebook demonstrates:
1. Connecting to `bodhan_mcp` over **stdio** using the official MCP Python client.
2. Path sandboxing, script detection, and **no-waste `DiskCache`** deduplication.
3. A **multi-step `BodhanAgentLoop`** pipeline with speech back-transcription verification and cost accounting.
4. **Honest refusal** on an impossible request (unsupported language) with ₹0 model spend.
5. Optional live CLI run when `.env` credentials are configured.

In [ ]:
import io
import json
import os
import sys
import tempfile
import wave
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

key_status = {
    name: bool((os.getenv(name) or "").strip())
    for name in (
        "BODHAN_TRANSCRIBE_KEY",
        "BODHAN_TRANSLATE_KEY",
        "BODHAN_SPEAK_KEY",
        "BODHAN_OCR_KEY",
        "LLM_API_KEY",
    )
}
print("Configured credentials (booleans only):", json.dumps(key_status, indent=2))

## 1. Standalone MCP Server (`python -m bodhan_mcp` over stdio)

Any third-party MCP client can connect to `bodhan_mcp` directly over stdio, discover typed schemas, invoke tools, and receive structured `ToolError` objects on invalid inputs.

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_params = StdioServerParameters(command=sys.executable, args=["-m", "bodhan_mcp"])

async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()

        tools_resp = await session.list_tools()
        print("Registered MCP tools:", [t.name for t in tools_resp.tools])

        caps = await session.call_tool("list_capabilities", {})
        caps_data = json.loads(caps.content[0].text)
        print("Supported languages:", caps_data["data"]["supported_languages"])
        print("Bengali default voices:", caps_data["data"]["default_voices"]["bn"])

        lang_res = await session.call_tool("detect_language", {"text_or_file": "నమస్కారం తెలుగు"})
        print("Telugu script check:", json.loads(lang_res.content[0].text)["data"])

        err_res = await session.call_tool("transcribe_audio", {"file_path": "does_not_exist.wav"})
        print("Structured error on missing file:", json.loads(err_res.content[0].text))

## 2. Path Sandboxing, Script Ambiguity & `DiskCache` Deduplication

Tools never pass media inline as base64 in LLM context, block path traversal outside the workspace, and deduplicate repeated `(model, params, input_hash)` calls.

In [ ]:
from bodhan_mcp.cache import DiskCache
from bodhan_mcp.client import BodhanClient
from bodhan_mcp.models import ToolError, Usage
from bodhan_mcp.tools.detect_language import detect_language
from bodhan_mcp.tools.translate import translate_text
from bodhan_mcp.workspace import Workspace


def make_wav_bytes(seconds: float = 0.2) -> bytes:
    buf = io.BytesIO()
    with wave.open(buf, "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(8000)
        wf.writeframes(b"\x00\x00" * int(8000 * seconds))
    return buf.getvalue()


class DemoMockClient(BodhanClient):
    def __init__(self):
        self.calls = {"transcribe": 0, "translate": 0, "speak": 0, "ocr": 0}

    def transcribe(self, audio_bytes, filename, language=None):
        self.calls["transcribe"] += 1
        return {
            "text": "नमस्ते, यह एक परीक्षण है।",
            "language": language or "hi",
            "duration": 1.0,
            "usage": Usage(model="indic-transcribe", cost_inr=0.01),
        }

    def translate(self, text, target_language, source_language=None):
        self.calls["translate"] += 1
        return {
            "translated_text": f"[{target_language}] வணக்கம்",
            "usage": Usage(model="indic-translate", cost_inr=0.02),
        }

    def speak(self, text, language, voice_id=None):
        self.calls["speak"] += 1
        return {
            "audio_bytes": make_wav_bytes(0.2),
            "format": "wav",
            "usage": Usage(model="indic-speak", cost_inr=0.03),
        }


with tempfile.TemporaryDirectory() as tmpdir:
    ws = Workspace(tmpdir)
    cache = DiskCache(Path(tmpdir) / "cache")
    mock = DemoMockClient()

    try:
        ws.resolve_input("../etc/passwd")
    except ToolError as exc:
        print("Sandbox blocked traversal:", exc.code.value, exc.message)

    print("Devanagari ambiguity:", detect_language("नमस्ते दुनिया", ws)["data"])

    first = translate_text(ws, "Half is 1/2", "ta", client=mock, cache=cache)
    second = translate_text(ws, "Half is 1/2", "ta", client=mock, cache=cache)
    print("API calls made for 2 identical requests:", mock.calls["translate"])
    print("1st usage:", first["usage"])
    print("2nd usage (cached):", second["usage"])

## 3. Multi-Step Pipeline with Verification & Cost Budgeting

Here `BodhanAgentLoop` runs a 3-step composable plan (`transcribe` → `translate` → `speak`) with automatic speech back-transcription verification and full step-level trace recording.

In [ ]:
from bodhan_agent.loop import BodhanAgentLoop

with tempfile.TemporaryDirectory() as tmpdir:
    root = Path(tmpdir)
    (root / "voice_note.wav").write_bytes(make_wav_bytes(1.0))

    mock = DemoMockClient()
    agent = BodhanAgentLoop(workspace_dir=root, client=mock, max_steps=10, max_cost_inr=5.0)

    plan = [
        {
            "id": "step_transcribe",
            "tool": "transcribe",
            "arguments": {"file_path": "voice_note.wav", "language": "hi"},
        },
        {
            "id": "step_translate",
            "tool": "translate",
            "arguments": {
                "target_language": "ta",
                "text": {"$ref": {"step_id": "step_transcribe", "output": "transcript"}},
            },
        },
        {
            "id": "step_speak",
            "tool": "speak",
            "arguments": {
                "language": "ta",
                "voice_gender": "female",
                "text": {"$ref": {"step_id": "step_translate", "output": "translation"}},
            },
        },
    ]
    deliverables = [
        {
            "name": "tamil_audio",
            "step_id": "step_speak",
            "output": "audio_path",
            "media_type": "audio",
            "description": "Tamil synthesized speech output.",
        }
    ]

    result = agent.execute_pipeline(plan, deliverables=deliverables)
    print("Pipeline success:", result.success)
    print("Total cost (INR):", result.total_cost_inr)
    print("Verification:", json.dumps(result.verification, indent=2))
    print("Assumptions:", result.assumptions)
    print("Trace event sequence:", [ev["event"] for ev in result.trace])

## 4. Honest Failure on Impossible Tasks

Unsupported languages (such as French) are rejected during preflight / plan validation before any paid Bodhan call is constructed.

In [ ]:
from bodhan_agent.preflight import validate_plan
from bodhan_agent.tool_catalog import build_tool_catalog

with tempfile.TemporaryDirectory() as tmpdir:
    ws = Workspace(tmpdir)
    validation = validate_plan(
        [],
        ws,
        build_tool_catalog({"BODHAN_TRANSLATE_KEY": "configured"}),
        refusal={
            "code": "UNSUPPORTED_LANGUAGE",
            "message": "French ('fr') is outside Bodhan's supported Indian-language catalog.",
            "hint": "Specify one of the 23 configured languages (e.g., hi, ta, bn, te, mr).",
        },
    )
    print("Refusal validation result:", json.dumps(validation, indent=2))

## 5. Optional Live CLI Execution

If you have `BODHAN_TRANSLATE_KEY` and `LLM_API_KEY` configured in `.env`, this cell invokes the real `python -m bodhan_agent` CLI on a small temporary text input and prints the generated `report.json`.

In [ ]:
import subprocess

if key_status["BODHAN_TRANSLATE_KEY"] and key_status["LLM_API_KEY"]:
    with tempfile.TemporaryDirectory() as tmpdir:
        tmp = Path(tmpdir)
        sample = tmp / "note.txt"
        sample.write_text("The meeting has been postponed to Monday at 10 AM.", encoding="utf-8")
        out_dir = tmp / "out"
        proc = subprocess.run(
            [
                sys.executable,
                "-m",
                "bodhan_agent",
                "--task",
                "Translate this note into Tamil",
                "--files",
                str(sample),
                "--out",
                str(out_dir),
                "--max-steps",
                "5",
                "--max-cost-inr",
                "1.0",
            ],
            capture_output=True,
            text=True,
        )
        print("Exit code:", proc.returncode)
        print((out_dir / "result.md").read_text(encoding="utf-8"))
else:
    print("Skipping live CLI call (set BODHAN_TRANSLATE_KEY and LLM_API_KEY in .env to run).")